# Week 29, Day 2: Equality and hashing

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w029d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

a, b = Point(1, 2), Point(1, 2)
print(a == b, a is b, a == a)
print(len({a, b}), [1, 2] == [1, 2], len({(1, 2), (1, 2)}))

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Point({self.x!r}, {self.y!r})"

    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

a, b = Point(1, 2), Point(1, 2)
print(a == b, a != b, a is b)
print(a == (1, 2), a == "Point(1, 2)", Point(1, 2) in [Point(0, 0), Point(1, 2)])

**Predict before you run:** what will this print?



In [ ]:
print(hash(42) == hash(42), hash("abc") == hash("abc"), hash((1, 2)) == hash((1, 2)))
print(hash(1) == hash(1.0) == hash(True))
try:
    hash([1, 2])
except TypeError as e:
    print("TypeError:", e)

**Predict before you run:** what will this print?



In [ ]:
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

try:
    {Point(1, 2)}
except TypeError as e:
    print("TypeError:", e)

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f"Point({self.x!r}, {self.y!r})"

    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self):
        return hash((self.x, self.y))

visited = {Point(0, 0), Point(1, 2), Point(1, 2)}
print(visited)
heights = {Point(0, 0): 12, Point(1, 2): 30}
print(heights[Point(1, 2)], Point(5, 5) in heights)

**Predict before you run:** what will this print?



In [ ]:
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __eq__(self, other):
        return isinstance(other, Point) and (self.x, self.y) == (other.x, other.y)
    def __hash__(self):
        return hash((self.x, self.y))

p = Point(1, 2)
seen = {p}
p.x = 99
print(p in seen, Point(1, 2) in seen, Point(99, 2) in seen, len(seen))

> **Key idea.** `__eq__` defines `==`: compare tuples of the attributes that matter, and return `NotImplemented` for other types; `!=` follows automatically. Equal objects must have equal hashes, so defining `__eq__` removes the default hash. For value-like objects, write `__hash__` hashing the same tuple, and never change those attributes afterwards. Objects with a changing identity-like state should stay unhashable, or compare by identity.

### Exercise: Equal amounts

Give `Money` an `__eq__` (equal when the pence are equal; `NotImplemented` for anything that isn't `Money`) and a matching `__hash__`.

In [ ]:
class Money:
    def __init__(self, pence):
        self.pence = pence

    def __repr__(self):
        return f"Money({self.pence})"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"equality and hashing\"\n    M = ns[\"Money\"]\n    assert M(500) == M(500) and M(500) != M(501) and not (M(500) == 500) and M(5) != \"5\"\n    assert len({M(1), M(1), M(2)}) == 2 and {M(7): \"x\"}[M(7)] == \"x\"\n    assert M(500).__eq__(500) is NotImplemented")

### Exercise: Playing cards

Write `Card(rank, suit)` where rank is one of `"2"` to `"10"`, `"J"`, `"Q"`, `"K"`, `"A"` (case-insensitive: `"q"` becomes `"Q"`) and suit is one of `"♠♥♦♣"` (or the letters `S H D C`, any case, which become the symbols). Bad values raise `ValueError`. Cards are equal when rank and suit match, and hashable. `__repr__` gives `Card('Q', '♥')`.

In [ ]:
class Card:
    RANKS = ["2", "3", "4", "5", "6", "7", "8", "9", "10", "J", "Q", "K", "A"]
    SUITS = {"S": "♠", "H": "♥", "D": "♦", "C": "♣"}
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"cards\"\n    C = ns[\"Card\"]\n    assert C(\"q\", \"h\") == C(\"Q\", \"\u2665\") and repr(C(\"10\", \"s\")) == \"Card('10', '\u2660')\"\n    assert C(\"A\", \"\u2663\") != C(\"A\", \"\u2660\") and len({C(\"k\", \"d\"), C(\"K\", \"\u2666\"), C(\"K\", \"D\")}) == 1\n    hand = [C(\"2\", \"c\"), C(\"j\", \"h\")]\n    assert C(\"J\", \"\u2665\") in hand and hand.index(C(\"J\", \"H\")) == 1\n    for bad in [(\"1\", \"s\"), (\"Q\", \"x\"), (\"11\", \"h\")]:\n        try:\n            C(*bad)\n            assert False, f\"Card{bad} should raise ValueError\"\n        except ValueError:\n            pass")

### Exercise: Equal, ignoring case

`Word(text)` stores the text as given (for display), but two words are equal if their texts match **ignoring case and spaces at the ends**. Write `__eq__` and `__hash__` so that equal words have equal hashes, then use a set to write `distinct(texts)` returning how many different words there are.

In [ ]:
class Word:
    def __init__(self, text):
        self.text = text

    def __repr__(self):
        return f"Word({self.text!r})"

def distinct(texts):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"values\"\n    W = ns[\"Word\"]\n    assert W(\"Hello\") == W(\" hello \") and W(\"Hello\").text == \"Hello\" and hash(W(\"ABC\")) == hash(W(\"abc \"))\n    assert distinct([\"Apple\", \"apple\", \" APPLE\", \"pear\", \"Pear \"]) == 2 and distinct([]) == 0\n    assert W(\"x\") != \"x\"")

### Exercise: Points as keys

Using the hashable `Cell(row, col)` in the starter, write `count_neighbours(alive)`: given a set of live cells, return a dictionary from each cell that has **at least one** live neighbour (the 8 around it, not itself) to how many live neighbours it has. This is the heart of Conway's Game of Life.

In [ ]:
class Cell:
    def __init__(self, row, col):
        self.row, self.col = row, col

    def __repr__(self):
        return f"Cell({self.row}, {self.col})"

    def __eq__(self, other):
        if not isinstance(other, Cell):
            return NotImplemented
        return (self.row, self.col) == (other.row, other.col)

    def __hash__(self):
        return hash((self.row, self.col))

def count_neighbours(alive):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"counts\"\n    C = ns[\"Cell\"]\n    alive = {C(0, 0), C(0, 1), C(0, 2)}\n    n = count_neighbours(alive)\n    assert n[C(0, 1)] == 2 and n[C(0, 0)] == 1 and n[C(1, 1)] == 3 and n[C(-1, -1)] == 1 and C(0, 4) not in n\n    assert len(n) == 15 and count_neighbours(set()) == {}")

### Bug hunt: Unhashable

Adding a `Colour` to a set crashes with `TypeError: unhashable type: 'Colour'`. Colours never change once made. Fix the class so they can go in sets.

In [ ]:
class Colour:
    def __init__(self, r, g, b):
        self.r, self.g, self.b = r, g, b

    def __eq__(self, other):
        if not isinstance(other, Colour):
            return NotImplemented
        return (self.r, self.g, self.b) == (other.r, other.g, other.b)

palette = {Colour(255, 0, 0), Colour(255, 0, 0), Colour(0, 0, 255)}
print(len(palette))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"two colours\"\n    expect_output(\"2\")")

### Bug hunt: Comparing with None

`find` crashes with `AttributeError: 'NoneType' object has no attribute 'code'` because the list has a `None` in it. Fix `__eq__` (not `find`) so comparing a `Product` with anything else simply isn't equal.

In [ ]:
class Product:
    def __init__(self, code, name):
        self.code, self.name = code, name

    def __eq__(self, other):
        return self.code == other.code

    def __hash__(self):
        return hash(self.code)

def find(items, wanted):
    return [i for i, item in enumerate(items) if item == wanted]

stock = [Product("A1", "pen"), None, Product("B2", "ink"), Product("A1", "pen (blue)")]
print(find(stock, Product("A1", "?")))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"finds both\"\n    expect_output(\"[0, 3]\")\n    P = ns[\"Product\"]\n    assert P(\"A1\", \"x\").__eq__(None) is NotImplemented")

### Bug hunt: Equal but different hashes

Two tags that are equal end up as two items in a set, because `__hash__` and `__eq__` disagree. Fix `__hash__`.

In [ ]:
class Tag:
    def __init__(self, name):
        self.name = name

    def __eq__(self, other):
        if not isinstance(other, Tag):
            return NotImplemented
        return self.name.lower() == other.name.lower()

    def __hash__(self):
        return hash(self.name)

print(len({Tag("Python"), Tag("python"), Tag("PYTHON")}))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"one tag\"\n    expect_output(\"1\")")

### Exercise: Predict them



In [ ]:
class K:
    def __init__(self, v):
        self.v = v
    def __eq__(self, other):
        return isinstance(other, K) and self.v == other.v
    def __hash__(self):
        return hash(self.v)

class N:
    def __init__(self, v):
        self.v = v

p = ___   # K(1) == K(1)
q = ___   # N(1) == N(1)
r = ___   # len({K(1), K(1), K(2)})
s = ___   # len({N(1), N(1)})
t = ___   # K(1) is K(1)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r, s, t) == (True, False, 2, 2, False)")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: version numbers

Write `Version(text)` for version strings like `"1.10.2"`: one or more whole numbers separated by dots (`ValueError` otherwise, including empty parts and negative numbers). Trailing zeros don't matter: `Version("1.2") == Version("1.2.0") == Version("1.2.0.0")`, and their hashes must match. Keep the original text for `__repr__` (`Version('1.2.0')`) but compare by the numbers. Also write `newest(texts)` returning the text of the highest version (numbers compared part by part; `None` for no texts), keeping the first if equal versions appear.

In [ ]:
import re

class Version:
    ___

def newest(texts):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_equal():\n    \"equality and hashing\"\n    V = ns[\"Version\"]\n    assert V(\"1.2\") == V(\"1.2.0\") == V(\"1.2.0.0\") and V(\"1.2\") != V(\"1.20\") and V(\"0\") == V(\"0.0\")\n    assert len({V(\"2\"), V(\"2.0\"), V(\"2.0.1\")}) == 2 and repr(V(\"1.2.0\")) == \"Version('1.2.0')\"\n    assert V(\"1.0\").__eq__(\"1.0\") is NotImplemented\ndef test_bad():\n    \"bad text\"\n    V = ns[\"Version\"]\n    for t in [\"\", \"1..2\", \"1.2.\", \"v1.2\", \"1.-2\", \"1.2a\"]:\n        try:\n            V(t)\n            assert False, f\"Version({t!r}) should raise ValueError\"\n        except ValueError:\n            pass\ndef test_newest():\n    \"newest\"\n    assert newest([\"1.9\", \"1.10\", \"1.9.9\", \"0.99\"]) == \"1.10\"\n    assert newest([\"2.0\", \"2\", \"1.99\"]) == \"2.0\" and newest([]) is None")

## Warm-up

Without any special methods, `==` falls back to `is`: two points with the same coordinates aren't equal, and a set keeps both. Lists and tuples compare their contents, and a set of equal tuples keeps one. Today your classes learn to do the same.

## Teaching == what equal means

`a == b` calls `a.__eq__(b)`. Write it to compare whatever makes two objects "the same thing" for your class, usually by comparing tuples of the attributes that matter:

- **`NotImplemented`** (a special value, not the error `NotImplementedError`) means "I don't know how to compare with that". Python then tries the other object's `__eq__`, and if neither knows, falls back to `is`, giving False. Returning it for other types is better than crashing on `other.x`, or deciding on behalf of a type you don't know about.
- **`!=`** comes for free: Python inverts `__eq__` unless you write `__ne__` yourself (you almost never should).
- **`in`** for lists, `list.index`, `list.count` and `remove` all use `==`, so they now work by value.
- `is` still means "the very same object". Use `is` only for identity, mostly `x is None`.

Decide carefully what "equal" means. Two bank accounts with the same owner and balance are probably **not** the same account; two points with the same coordinates are the same point. Objects that represent a **value** (a point, an amount of money, a date) should compare by value; objects that represent a **thing with an identity** (an account, a player, a lift) are often best left comparing by identity.

## What a hash is for

Sets and dictionaries find things fast (Week 17: O(1) on average) by **hashing**: they turn each key into a whole number, its **hash**, and use it to jump straight to where the key should be stored. `hash(x)` shows it.

A set looks for a key only among the objects with the **same hash**, then checks those with `==`. So there's one rule every hashable object must follow:

> **If `a == b`, then `hash(a) == hash(b)`.**

(The reverse needn't hold: different objects may share a hash. That's a "collision", and only slows things down a little.) That's why `1`, `1.0` and `True`, which are all equal, have the same hash. And it's why lists, which can change, have no hash at all: if a list's contents changed while it was in a set, its hash would change, and the set would look for it in the wrong place.

## Writing __hash__

When a class defines `__eq__`, Python **removes** its default hash (sets `__hash__` to `None`), because the default hash is based on identity and would break the rule. So a class with only `__eq__` can't go in a set:

If instances should be usable in sets and as keys, write `__hash__` to hash **the same attributes `__eq__` compares**, usually by hashing a tuple of them:

Using the same tuple in both methods guarantees the rule: equal points have equal tuples, so equal hashes.

Should a class be hashable? Only if its equality can't change while it's in use: values like points, money, dates, versions, playing cards. For objects whose state changes (a basket, an account), either leave the default identity-based `==` and hash (don't write `__eq__`), or write `__eq__` and leave them unhashable.

## The mutable key trap

If a hashable object's attributes **can** be changed, changing them while it's in a set or a dictionary key breaks things silently:

The set stored `p` under its **old** hash. Now `p` is looked for under its new hash and isn't found; the old position holds an object that's no longer equal to `Point(1, 2)`. The object is in the set, and can't be found by any lookup.

The rule: **never change the attributes a hash depends on** while the object is in a set or used as a key. The safest way is to make such objects **immutable**: set their attributes once in `__init__` and never change them. Week 29, Day 4 (properties) and Week 32 (frozen dataclasses) show ways to enforce that.

## Practice

## Quiz

## Daily challenge

## Before you go

For each class you've written, decide: does it represent a value (compare by value, hashable, never changed) or a thing with an identity (compare by identity, or by value but unhashable)? Tomorrow: operator overloading, so `Money(5) + Money(3)` and `Vector(1, 2) * 3` work.